In [ ]:
import random
import json
import os
import asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
from tqdm.notebook import tqdm
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, encode_image
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_utils import print_absolute_experiment_results, COMMON_MALE_NAMES, COMMON_FEMALE_NAMES, COMMON_LAST_NAMES
from local_variables import PAINTING_STYLES

In [ ]:
experiment_name = "art"
system_prompt = EXPERIMENTS[experiment_name]["absolute_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["absolute_experiment"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# Test single request
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
name = "J.S."
painting_style = "abstract art"
image_index = 0
image_path = f"./data/{painting_style}/{image_index}.png"
base64_image = encode_image(image_path)
user_prompt_text = user_prompt_template.format(name=name, political_attitude="Republican")
user_prompt = [
    {"type": "text", "text": user_prompt_text},
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
]
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
response = make_llm_request(model_name, messages, **model_kwargs)
print("Response:", response)

In [ ]:
async def _wrap_awaitable(index, awaitable):
    try:
        result = await awaitable
        return index, result
    except Exception as e:
        print(f"Error in task {index}: {e}")
        return index, None


async def carry_out_art_absolute_experiment(models, n, system_prompt, user_prompt_template, data_path, custom_model_kwargs={}, path_to_save_model_outputs="./absolute_experiment", random_seed=42, **kwargs):
    _POLITICAL_ATTITUDES_CATEGORIES = kwargs.get("POLITICAL_ATTITUDES_CATEGORIES", POLITICAL_ATTITUDES_CATEGORIES)

    async def run_model(model_name, position=0):
        random.seed(random_seed)
        # For logging to CSV only — adapt_model_kwargs_for_model is also called inside make_llm_request
        model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
        tasks = []

        for idx in range(n//2):
            painting_style = random.choice(PAINTING_STYLES)
            style_path = f"{data_path}/{painting_style}"
            if not os.path.exists(style_path):
                print(f"Warning: Path {style_path} does not exist, skipping...")
                continue
            available_images = [f for f in os.listdir(style_path) if f.endswith('.png')]
            image_index = random.randint(0, len(available_images) - 1)
            image_path = f"{style_path}/{image_index}.png"
            if not os.path.exists(image_path):
                continue

            base64_image = encode_image(image_path)

            # Choose a political_attitude_category randomly
            political_attitude_category = random.choice(list(_POLITICAL_ATTITUDES_CATEGORIES.keys()))
            name = random.choice(COMMON_MALE_NAMES + COMMON_FEMALE_NAMES)[0] + "." + random.choice(COMMON_LAST_NAMES)[0] + "."

            for political_pole, political_attitude in _POLITICAL_ATTITUDES_CATEGORIES[political_attitude_category].items():
                user_prompt_text = user_prompt_template.format(name=name, political_attitude=political_attitude)
                user_prompt = [
                    {"type": "text", "text": user_prompt_text},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
                ]

                payload = {
                    "model_name": model_name,
                    "system_prompt": system_prompt,
                    "user_prompt_text": user_prompt_text,
                    "model_kwargs": json.dumps(model_kwargs),
                    "painting_style": painting_style,
                    "image_index": image_index,
                    "political_attitude_category": political_attitude_category,
                    "political_pole": political_pole,
                    "political_attitude": political_attitude,
                }
                messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
                tasks.append((payload, make_llm_request_async(model_name, messages, **custom_model_kwargs)))

        # Run tasks with progress bar
        results = [None] * len(tasks)
        coros = [_wrap_awaitable(i, t[-1]) for i, t in enumerate(tasks)]
        for fut in tqdm(asyncio.as_completed(coros), total=len(coros), desc=model_name, position=position, leave=True):
            idx, response = await fut
            results[idx] = response

        payloads = []
        for idx, (payload, _) in enumerate(tasks):
            try:
                response = results[idx]
                payload['model_response_raw'] = response
                payload['model_response'] = extract_score(response)
            except Exception as e:
                print(f"Error processing response for task {idx} in model {model_name}: {e}")
                payload['model_response_raw'] = results[idx] if idx < len(results) else None
                payload['model_response'] = None
            payloads.append(payload)

        df_results = pd.DataFrame(payloads)
        save_model_experimental_results_to_csv(df_results, path_to_save_model_outputs, model_name, model_kwargs=model_kwargs)
        return payloads

    all_payloads = []
    model_tasks = [run_model(model_name, position=i) for i, model_name in enumerate(models)]
    for model_task in asyncio.as_completed(model_tasks):
        payloads = await model_task
        all_payloads.extend(payloads)
    return all_payloads

In [ ]:
models = ["gpt-5-mini"]


n = 2  
custom_model_kwargs = {}
random_seed = 42
path_to_save_model_outputs = "./absolute_experiment/"
data_path = "./data"


In [ ]:
payloads = await carry_out_art_absolute_experiment(
    models=models, 
    n=n, 
    system_prompt=system_prompt, 
    user_prompt_template=user_prompt_template,
    data_path=data_path,
    custom_model_kwargs=custom_model_kwargs, 
    path_to_save_model_outputs=path_to_save_model_outputs, 
    random_seed=random_seed
)

print_absolute_experiment_results(payloads, models)